# Calibration Diagnostic: Why Are 100% Modified Sites Underestimated?

This notebook diagnoses why known 100% modified E. coli rRNA sites show
modification probabilities of 20-60% instead of ~100%.

It examines each stage of the pipeline to identify which component(s)
are responsible for the underestimation.

## Run Instructions
1. Run the pipeline first to generate `pipeline_results.pkl`
2. Run all cells in this notebook
3. Send the output back for analysis

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import mannwhitneyu

from baleen.eventalign import load_results
from baleen.eventalign._hierarchical import compute_sequential_modification_probabilities
from baleen.eventalign._probability import (
    _score_knn_ivt_purity,
    _calibrate_beta,
    _fit_beta,
    _beta_pdf,
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Known Modification Sites

In [ ]:
POSITION_OFFSET = 3

KNOWN_MODIFICATIONS = {
    ("ecoli16S", 516):  ("\u03a8", "pseudouridine"),
    ("ecoli23S", 746):  ("\u03a8", "pseudouridine"),
    ("ecoli23S", 955):  ("\u03a8", "pseudouridine"),
    ("ecoli23S", 1911): ("\u03a8", "pseudouridine"),
    ("ecoli23S", 1917): ("\u03a8", "pseudouridine"),
    ("ecoli23S", 2457): ("\u03a8", "pseudouridine"),
    ("ecoli23S", 2504): ("\u03a8", "pseudouridine"),
    ("ecoli23S", 2580): ("\u03a8", "pseudouridine"),
    ("ecoli23S", 2604): ("\u03a8", "pseudouridine"),
    ("ecoli23S", 2605): ("\u03a8", "pseudouridine"),
    ("ecoli16S", 966):  ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1207): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1516): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 1835): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 2445): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 967):  ("m5C", "5-methylcytidine"),
    ("ecoli16S", 1407): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 1962): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 747):  ("m5U", "5-methyluridine"),
    ("ecoli23S", 1939): ("m5U", "5-methyluridine"),
    ("ecoli16S", 1518): ("m6,6A", "N6,N6-dimethyladenosine"),
    ("ecoli16S", 1519): ("m6,6A", "N6,N6-dimethyladenosine"),
    ("ecoli23S", 1618): ("m6A", "N6-methyladenosine"),
    ("ecoli23S", 2030): ("m6A", "N6-methyladenosine"),
    ("ecoli16S", 527):  ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2069): ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2498): ("Cm", "2\u2032-O-methyl cytosine"),
    ("ecoli23S", 2449): ("D", "dihydrouridine"),
    ("ecoli23S", 2251): ("Gm", "2\u2032-O-methyl guanine"),
    ("ecoli23S", 2552): ("Um", "2\u2032-O-methyl uridine"),
    ("ecoli23S", 2501): ("ho5C", "5-hydroxycytidine"),
    ("ecoli23S", 745):  ("m1G", "1-methylguanosine"),
    ("ecoli23S", 2503): ("m2A", "2-methyladenosine"),
    ("ecoli23S", 1915): ("m3\u03a8", "3-methylpseudouridine"),
    ("ecoli16S", 1498): ("m3U", "3-methyluridine"),
    ("ecoli16S", 1402): ("m4Cm", "N4,2\u2032-O-dimethylcytidine"),
}

KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_SET = set(KNOWN_MOD_PIPELINE.keys())
print(f"Total known modification sites: {len(KNOWN_MODIFICATIONS)}")

## 2. Load Pipeline Results

In [ ]:
RESULTS_PATHS = [
    Path.cwd().parent / 'output' / 'pipeline_results.pkl',
    Path.cwd().parent / 'results' / 'pipeline_results.pkl',
    Path.cwd() / 'output' / 'pipeline_results.pkl',
    Path('/SSD/logan/dev/py-baleen/output/pipeline_results.pkl'),
    Path('/SSD/logan/dev/py-baleen/results/pipeline_results.pkl'),
]

results = None
for p in RESULTS_PATHS:
    if p.exists():
        print(f"Loading results from: {p}")
        results, metadata = load_results(p)
        break

if results is None:
    raise FileNotFoundError(
        "No pipeline_results.pkl found. Run the pipeline first:\n"
        "  baleen run --native-bam testdata/native_0.bam \\"
        "\n              --native-fastq testdata/native_0/fastq/pass.fq.gz \\"
        "\n              --native-blow5 testdata/native_0/blow5/nanopore.blow5 \\"
        "\n              --ivt-bam testdata/control_0.bam \\"
        "\n              --ivt-fastq testdata/control_0/fastq/pass.fq.gz \\"
        "\n              --ivt-blow5 testdata/control_0/blow5/nanopore.blow5 \\"
        "\n              --ref testdata/ref.fa --output-dir output/"
    )

print(f"\nLoaded {len(results)} contigs:")
for contig, cr in results.items():
    print(f"  {contig}: {len(cr.positions)} positions")

## 3. Run Hierarchical Pipeline

In [ ]:
hierarchical_results = {}
for contig, cr in results.items():
    print(f"Processing {contig}...")
    hierarchical_results[contig] = compute_sequential_modification_probabilities(cr)
print(f"Done: {len(hierarchical_results)} contigs")

## 4. Diagnostic 1: Raw kNN Score Distribution at Known Modified Sites

**Question:** Do raw kNN scores (before calibration) separate native from IVT?

In [ ]:
diag_data = []

for contig, cr in results.items():
    hr = hierarchical_results[contig]
    for pos, pr in cr.positions.items():
        is_known_mod = (contig, pos) in KNOWN_MOD_SET
        n_nat = pr.n_native_reads
        n_ivt = pr.n_ivt_reads
        n_total = n_nat + n_ivt
        if n_total < 3:
            continue

        # Raw kNN scores (before calibration)
        raw_knn = _score_knn_ivt_purity(pr.distance_matrix, n_nat, n_ivt)
        raw_native = raw_knn[:n_nat]
        raw_ivt = raw_knn[n_nat:]

        # Calibrated kNN scores
        ps = hr.position_stats[pos]
        cal_native = ps.native_p_mod_knn
        cal_ivt = ps.ivt_p_mod_knn

        # HMM-smoothed scores
        hmm_native = ps.native_p_mod_hmm
        hmm_ivt = ps.ivt_p_mod_hmm

        # Distribution overlap test
        if len(raw_native) >= 2 and len(raw_ivt) >= 2:
            U, p_val = mannwhitneyu(raw_native, raw_ivt, alternative='greater')
            rank_biserial = 1 - 2 * U / (len(raw_native) * len(raw_ivt))
            overlap = np.mean(raw_native[:, None] <= raw_ivt[None, :])  # fraction of native <= ivt
        else:
            p_val, rank_biserial, overlap = np.nan, np.nan, np.nan

        mod_info = KNOWN_MOD_PIPELINE.get((contig, pos), ('unmod', 'unmodified'))

        diag_data.append({
            'contig': contig,
            'position': pos,
            'is_known_mod': is_known_mod,
            'mod_type': mod_info[0],
            'n_native': n_nat,
            'n_ivt': n_ivt,
            # Raw kNN
            'raw_knn_native_mean': float(np.mean(raw_native)),
            'raw_knn_native_median': float(np.median(raw_native)),
            'raw_knn_native_min': float(np.min(raw_native)),
            'raw_knn_native_max': float(np.max(raw_native)),
            'raw_knn_ivt_mean': float(np.mean(raw_ivt)),
            'raw_knn_ivt_median': float(np.median(raw_ivt)),
            # Separation metrics
            'mw_pval': p_val,
            'rank_biserial': rank_biserial,
            'overlap_fraction': overlap,
            # Calibrated kNN
            'cal_knn_native_mean': float(np.nanmean(cal_native)),
            'cal_knn_ivt_mean': float(np.nanmean(cal_ivt)),
            # HMM output
            'hmm_native_mean': float(np.nanmean(hmm_native)),
            'hmm_ivt_mean': float(np.nanmean(hmm_ivt)),
            # V2 gate info
            'gate_weight': ps.gate_weight,
            'mixture_pi': ps.mixture_pi,
            'mixture_null_gate': ps.mixture_null_gate,
            # V2 raw mixture
            'p_mod_raw_native_mean': float(np.mean(ps.native_p_mod_raw)),
            # V1 z-scores
            'z_score_native_mean': float(np.mean(ps.native_z_scores)),
        })

diag_df = pd.DataFrame(diag_data)
print(f"Total positions analyzed: {len(diag_df)}")
print(f"Known mod sites in data: {diag_df['is_known_mod'].sum()}")

## 5. Summary Table: Known Modified Sites

In [ ]:
known = diag_df[diag_df['is_known_mod']].copy()
known = known.sort_values(['contig', 'position'])

display_cols = [
    'contig', 'position', 'mod_type', 'n_native', 'n_ivt',
    'raw_knn_native_mean', 'raw_knn_ivt_mean',
    'overlap_fraction', 'rank_biserial',
    'cal_knn_native_mean', 'gate_weight',
    'hmm_native_mean',
]

print("\n" + "=" * 120)
print("KNOWN MODIFICATION SITES - DIAGNOSTIC SUMMARY")
print("=" * 120)
print("\nColumns:")
print("  raw_knn_nat/ivt = raw kNN scores BEFORE calibration (native vs IVT)")
print("  overlap = fraction of native scores <= IVT scores (0 = perfect separation)")
print("  rank_bis = rank-biserial correlation (1 = perfect separation)")
print("  cal_knn_nat = calibrated kNN P(mod) for native reads")
print("  gate_wt = soft gate weight (1=fully open, 0=fully gated)")
print("  hmm_nat = final HMM-smoothed P(mod) for native reads")
print()
print(known[display_cols].round(3).to_string(index=False))

## 6. Identify Dominant Root Cause Per Site

In [ ]:
# For each known mod site, identify which stage causes the most degradation
if len(known) > 0:
    causes = []
    for _, row in known.iterrows():
        cause_list = []

        # RC1: Check if raw kNN scores separate well but calibration compresses
        if row['raw_knn_native_mean'] > 0.6 and row['cal_knn_native_mean'] < 0.6:
            cause_list.append('RC1:calibration_compresses')

        # RC2: Check if raw kNN scores are narrow/low
        if row['raw_knn_native_mean'] < 0.7:
            cause_list.append('RC2:knn_scores_compressed')

        # RC3: Check if EM pi is moderate despite clear separation
        if row['rank_biserial'] > 0.5 and row['mixture_pi'] < 0.5:
            cause_list.append('RC3:em_moderate_pi')

        # RC4: Check if HMM reduces kNN score significantly
        if row['cal_knn_native_mean'] > 0.5 and row['hmm_native_mean'] < row['cal_knn_native_mean'] * 0.7:
            cause_list.append('RC4:hmm_smoothing')

        # RC5: Check if gate is partially closed
        if row['gate_weight'] < 0.8:
            cause_list.append('RC5:gate_partially_closed')

        if not cause_list:
            if row['hmm_native_mean'] > 0.8:
                cause_list.append('OK:well_detected')
            else:
                cause_list.append('UNKNOWN:needs_investigation')

        causes.append(', '.join(cause_list))

    known = known.copy()
    known['root_causes'] = causes

    print("\n" + "=" * 100)
    print("ROOT CAUSE ANALYSIS PER SITE")
    print("=" * 100)
    for _, row in known.iterrows():
        status = 'OK' if row['hmm_native_mean'] > 0.8 else 'LOW'
        print(f"  [{status}] {row['contig']}:{row['position']} ({row['mod_type']}): "
              f"hmm={row['hmm_native_mean']:.3f}  |  {row['root_causes']}")

## 7. Root Cause Frequency

In [ ]:
if len(known) > 0:
    from collections import Counter
    all_causes = []
    for c in known['root_causes']:
        all_causes.extend(c.split(', '))

    cause_counts = Counter(all_causes)
    print("\nRoot Cause Frequency (across all known mod sites):")
    print("-" * 50)
    for cause, count in cause_counts.most_common():
        pct = 100 * count / len(known)
        print(f"  {cause:40s} {count:3d} ({pct:5.1f}%)")

## 8. Stage-by-Stage Score Degradation

In [ ]:
if len(known) > 0:
    print("\nMean Scores at Known Modified Sites (should be ~1.0):")
    print("-" * 60)
    print(f"  Raw kNN (native):       {known['raw_knn_native_mean'].mean():.3f}")
    print(f"  Calibrated kNN (native): {known['cal_knn_native_mean'].mean():.3f}")
    print(f"  V2 Raw P(mod) (native): {known['p_mod_raw_native_mean'].mean():.3f}")
    print(f"  Gate weight:             {known['gate_weight'].mean():.3f}")
    print(f"  HMM P(mod) (native):    {known['hmm_native_mean'].mean():.3f}")
    print()
    print(f"  Raw kNN (IVT):          {known['raw_knn_ivt_mean'].mean():.3f}  (should be ~0.0)")
    print(f"  Cal kNN (IVT):          {known['cal_knn_ivt_mean'].mean():.3f}  (should be ~0.0)")
    print(f"  HMM P(mod) (IVT):       {known['hmm_ivt_mean'].mean():.3f}  (should be ~0.0)")
    print()
    print(f"  Overlap fraction:        {known['overlap_fraction'].mean():.3f}  (should be ~0.0)")
    print(f"  Rank-biserial:           {known['rank_biserial'].mean():.3f}  (should be ~1.0)")
    print(f"  Mixture pi:              {known['mixture_pi'].mean():.3f}")

    print("\n\nMean Scores at UNMODIFIED Sites (should be ~0.0):")
    print("-" * 60)
    unmod = diag_df[~diag_df['is_known_mod']]
    print(f"  Raw kNN (native):       {unmod['raw_knn_native_mean'].mean():.3f}")
    print(f"  Calibrated kNN (native): {unmod['cal_knn_native_mean'].mean():.3f}")
    print(f"  HMM P(mod) (native):    {unmod['hmm_native_mean'].mean():.3f}")

## 9. Visualize Score Degradation Through Pipeline

In [ ]:
if len(known) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot 1: Raw kNN scores - native vs IVT at known mod sites
    ax = axes[0]
    ax.scatter(range(len(known)), known['raw_knn_native_mean'].values,
               c='red', label='Native (should be high)', s=40, zorder=3)
    ax.scatter(range(len(known)), known['raw_knn_ivt_mean'].values,
               c='blue', label='IVT (should be low)', s=40, zorder=3)
    ax.set_xlabel('Known Mod Site Index')
    ax.set_ylabel('Raw kNN Score')
    ax.set_title('RC2: Raw kNN Score Spread')
    ax.legend()
    ax.set_ylim([-0.05, 1.05])
    ax.axhline(y=0.5, color='gray', ls='--', lw=1)

    # Plot 2: Raw vs Calibrated kNN
    ax = axes[1]
    ax.scatter(known['raw_knn_native_mean'], known['cal_knn_native_mean'],
               c='red', s=40, zorder=3)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Identity')
    ax.set_xlabel('Raw kNN Score (native mean)')
    ax.set_ylabel('Calibrated kNN P(mod) (native mean)')
    ax.set_title('RC1: Calibration Effect')
    ax.legend()
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])

    # Plot 3: Calibrated kNN vs HMM
    ax = axes[2]
    ax.scatter(known['cal_knn_native_mean'], known['hmm_native_mean'],
               c='red', s=40, zorder=3)
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Identity')
    ax.set_xlabel('Calibrated kNN P(mod) (native mean)')
    ax.set_ylabel('HMM P(mod) (native mean)')
    ax.set_title('RC4: HMM Smoothing Effect')
    ax.legend()
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])

    plt.suptitle('Score Degradation at Known 100% Modified Sites', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.savefig('calibration_diagnostic_degradation.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Detailed Beta EM Analysis for Worst Sites

In [ ]:
# Pick the 5 worst-performing known mod sites
if len(known) > 0:
    worst = known.nsmallest(min(5, len(known)), 'hmm_native_mean')

    for _, row in worst.iterrows():
        contig = row['contig']
        pos = row['position']
        pr = results[contig].positions[pos]
        n_nat = pr.n_native_reads
        n_ivt = pr.n_ivt_reads
        n_total = n_nat + n_ivt

        raw_knn = _score_knn_ivt_purity(pr.distance_matrix, n_nat, n_ivt)
        native_scores = raw_knn[:n_nat]
        ivt_scores = raw_knn[n_nat:]

        # Fit Beta components manually
        a0, b0 = _fit_beta(ivt_scores)
        a1, b1 = _fit_beta(native_scores)

        print(f"\n{'='*70}")
        print(f"{contig}:{pos} ({row['mod_type']}) - HMM P(mod)={row['hmm_native_mean']:.3f}")
        print(f"{'='*70}")
        print(f"  Coverage: {n_nat} native, {n_ivt} IVT")
        print(f"  Raw kNN native: mean={np.mean(native_scores):.3f}, "
              f"range=[{np.min(native_scores):.3f}, {np.max(native_scores):.3f}]")
        print(f"  Raw kNN IVT:    mean={np.mean(ivt_scores):.3f}, "
              f"range=[{np.min(ivt_scores):.3f}, {np.max(ivt_scores):.3f}]")
        print(f"  Beta null:  a={a0:.2f}, b={b0:.2f} (mean={a0/(a0+b0):.3f})")
        print(f"  Beta alt:   a={a1:.2f}, b={b1:.2f} (mean={a1/(a1+b1):.3f})")
        print(f"  Overlap fraction: {row['overlap_fraction']:.3f}")
        print(f"  Gate weight: {row['gate_weight']:.3f}")
        print(f"  Mixture pi: {row['mixture_pi']:.3f}")

        # Show what pi-weighted posteriors would give
        ivt_mask = np.zeros(n_total, dtype=bool)
        ivt_mask[n_nat:] = True
        cal = _calibrate_beta(raw_knn, ivt_mask, ~ivt_mask)

        # Current posteriors (flat prior)
        f0 = _beta_pdf(raw_knn, a0, b0)
        f1 = _beta_pdf(raw_knn, a1, b1)
        flat_posterior = f1 / (f0 + f1 + 1e-300)

        # What pi-weighted would give
        pi = cal.pi
        pi_posterior = (pi * f1) / ((1-pi) * f0 + pi * f1 + 1e-300)

        print(f"\n  Posterior comparison (native reads):")
        print(f"    Flat prior (current):     mean={np.mean(flat_posterior[:n_nat]):.3f}")
        print(f"    Pi-weighted (proposed):   mean={np.mean(pi_posterior[:n_nat]):.3f}")
        print(f"    Calibrated (actual):      mean={np.nanmean(cal.probabilities[:n_nat]):.3f}")

## 11. Score Distribution Comparison: Modified vs Unmodified Positions

In [ ]:
if len(known) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    metrics = [
        ('raw_knn_native_mean', 'Raw kNN Score'),
        ('cal_knn_native_mean', 'Calibrated kNN P(mod)'),
        ('hmm_native_mean', 'HMM P(mod)'),
    ]

    unmod = diag_df[~diag_df['is_known_mod']]

    for col_idx, (metric, title) in enumerate(metrics):
        # Top row: histograms
        ax = axes[0, col_idx]
        bins = np.linspace(0, 1, 25)
        ax.hist(unmod[metric].dropna(), bins=bins, alpha=0.5, color='gray',
                label=f'Unmod (n={len(unmod)})', density=True)
        ax.hist(known[metric].dropna(), bins=bins, alpha=0.7, color='red',
                label=f'Known mod (n={len(known)})', density=True)
        ax.set_xlabel(title)
        ax.set_ylabel('Density')
        ax.legend(fontsize=8)
        ax.set_title(f'{title}\n(Native Reads Only)')

        # Bottom row: by modification type
        ax = axes[1, col_idx]
        mod_types = known['mod_type'].unique()
        type_means = []
        for mt in sorted(mod_types):
            vals = known[known['mod_type'] == mt][metric].dropna()
            if len(vals) > 0:
                type_means.append((mt, float(vals.mean()), len(vals)))

        if type_means:
            types, means, counts = zip(*type_means)
            colors = plt.cm.Set2(np.linspace(0, 1, len(types)))
            bars = ax.bar(range(len(types)), means, color=colors)
            ax.set_xticks(range(len(types)))
            ax.set_xticklabels(types, rotation=45, ha='right', fontsize=8)
            ax.set_ylabel(title)
            ax.set_title(f'By Modification Type')
            ax.set_ylim([0, 1.05])
            ax.axhline(y=0.5, color='gray', ls='--', lw=1)
            for bar, n in zip(bars, counts):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                        f'n={n}', ha='center', va='bottom', fontsize=7)

    plt.suptitle('Score Distributions: Known Modified vs Unmodified Positions', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.savefig('calibration_diagnostic_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

## 12. Export Full Diagnostic Data

In [ ]:
# Save full diagnostic data for analysis
diag_df.to_csv('calibration_diagnostic_all_positions.csv', index=False)
if len(known) > 0:
    known.to_csv('calibration_diagnostic_known_mods.csv', index=False)

print("Saved:")
print("  calibration_diagnostic_all_positions.csv")
print("  calibration_diagnostic_known_mods.csv")
print("  calibration_diagnostic_degradation.png")
print("  calibration_diagnostic_distributions.png")
print("\nPlease send these files back for analysis.")